In [1]:
import os
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from dotenv import load_dotenv
from tqdm import tqdm
import re
import unicodedata
import difflib
import signal, sys
from underthesea import ner


# ---------------- LOAD .env ----------------

In [2]:
dotenv_path = os.path.join(os.path.dirname(os.getcwd()), "..", ".env")
if os.path.exists(dotenv_path):
    load_dotenv(dotenv_path)
else:
    load_dotenv()
print("Loaded environment variables from .env file")


Loaded environment variables from .env file


# ---------------- CONFIG ----------------

In [3]:
TYPE_DATA = "test"  # test
COLUMNS_TO_FIX = ["prompt"]

MODEL = "chamdentimem/ViT5_Vietnamese_Correction"
# MODEL = "bmd1905/vietnamese-correction-v2"
INPUT_CSV = "/home/guest/Projects/CS221/data/vihallu-public-test.csv"
DATA_PREPROCESSED = "/home/guest/Projects/CS221/data/preprocessed"
os.makedirs(DATA_PREPROCESSED, exist_ok=True)

OUTPUT_CSV = os.path.join(
    DATA_PREPROCESSED,
    f"vihallu-{TYPE_DATA}-{MODEL.split('/')[-1]}-{COLUMNS_TO_FIX[0]}-preprocessed.csv",
)

# ===== THAY ĐỔI: Thêm BATCH_SIZE =====
BATCH_SIZE = 32  # Bạn có thể tăng lên 32 hoặc 64 nếu VRAM GPU cho phép

correction_cache = {}  # Cache này giờ ít tác dụng hơn, nhưng cứ để
SAMPLE_DEBUG_N = 30
MAX_INPUT_LENGTH = 1024

# Giữ lại các ngưỡng đã được cân bằng từ lần trước
ACCEPT_SIMILARITY_THRESHOLD = 0.88
LENGTH_CHANGE_ALLOWED_RATIO = 0.2
STRICT_BASE_SIMILARITY_THRESHOLD = 0.92
NUM_BEAMS = 8
REPEITION_PENALTY = 1.2


# ---------------- Device ----------------

In [4]:
print("CUDA available:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


CUDA available: True


# Ctrl+C handler

In [5]:
df_global = None


def signal_handler(sig, frame):
    print("\n\n[Ctrl+C] Đang lưu tiến trình...")
    if df_global is not None:
        df_global.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
        print(f"Đã lưu thành công vào {OUTPUT_CSV}")
    sys.exit(0)


signal.signal(signal.SIGINT, signal_handler)


<function _signal.default_int_handler(signalnum, frame, /)>

# ---------- tokenizer/model setup ----------

In [6]:
print(f"Loading tokenizer for {MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL)

print(f"Loading model {MODEL}...")
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL).to(device)
model.eval()


Loading tokenizer for chamdentimem/ViT5_Vietnamese_Correction...
Loading model chamdentimem/ViT5_Vietnamese_Correction...


T5ForConditionalGeneration(
  (shared): Embedding(36096, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(36096, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=768, out_features=3072, bias=False)
              (wo): Linear(in_features=3072, out_features=768, bias=False)
              (dropout): Dro

# ---------- Utilities ----------

In [7]:
def remove_diacritics_and_punct(s: str) -> str:
    s = unicodedata.normalize("NFD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    s = unicodedata.normalize("NFC", s)
    s = re.sub(r"[^\w\s]", "", s)
    return s.lower()


def string_similarity(a, b):
    return difflib.SequenceMatcher(None, a, b).ratio()


# <<< THÊM MỚI: Các hàm "Fact-Checking" >>>
def extract_numbers(text: str) -> set:
    """Trích xuất tất cả các chuỗi số từ văn bản."""
    return set(re.findall(r"\d+", text))


def extract_proper_nouns(text: str) -> set:
    """
    Trích xuất các danh từ riêng tiềm năng.
    Heuristic: là các từ viết hoa không nằm ở đầu câu.
    """
    proper_nouns = set()
    # Tách văn bản thành các câu một cách tương đối
    sentences = re.split(r"(?<=[.?!])\s+", text)
    for sentence in sentences:
        words = sentence.split()
        if not words:
            continue
        # Bỏ qua từ đầu tiên của câu, kiểm tra các từ còn lại
        for word in words[1:]:
            # istitle() kiểm tra xem từ có phải dạng viết hoa chữ cái đầu không
            if word.istitle():
                # Chuẩn hóa về chữ thường để so sánh
                proper_nouns.add(word.lower().strip(".,;:!?"))
    return proper_nouns


# ---------- Prompt ----------

In [8]:
# Dung cho model chamdiemtimem/ViT5_Vietnamese_Correction
def make_correction_prompt(original_text: str) -> str:
    return f"Sửa lỗi chính tả và ngữ pháp: {original_text}"


# def make_correction_prompt(original_text: str) -> str:
#     # Model BART (bmd1905) không cần prefix, chỉ nhận văn bản thô
#     return original_text


# ---------- Generation ----------

In [9]:
# def generate_from_model(prompt_text):
#     inputs = tokenizer(
#         prompt_text, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LENGTH
#     ).to(device)

#     input_len = inputs.input_ids.shape[1]
#     max_new_tokens = min(int(input_len * 1.5) + 20, MAX_INPUT_LENGTH)

#     with torch.no_grad():
#         outputs = model.generate(
#             **inputs,
#             max_new_tokens=max_new_tokens,
#             num_beams=5,
#             early_stopping=True,
#             no_repeat_ngram_size=3
#         )
#     corrected = tokenizer.decode(outputs[0], skip_special_tokens=True)
#     return corrected.strip()


# ===== THAY ĐỔI: Hàm này giờ xử lý BATCH =====

In [10]:
def generate_batch_from_model(text_batch: list[str]):
    # Tokenize cả batch. `padding=True` để đệm các câu ngắn
    # cho bằng câu dài nhất trong batch.
    inputs = tokenizer(
        text_batch,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=MAX_INPUT_LENGTH,
    ).to(device)

    # Ước tính max_new_tokens dựa trên câu dài nhất trong batch
    input_len = inputs.input_ids.shape[1]
    max_new_tokens = min(int(input_len * 1.5) + 20, MAX_INPUT_LENGTH)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=NUM_BEAMS,
            repetition_penalty=REPEITION_PENALTY,  # Vì ViT ko cần dùng, còn BART thì có
            early_stopping=True,
            no_repeat_ngram_size=3
        )

    # Giải mã cả batch.
    corrected_batch = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    # Trả về một list các string đã được sửa
    return [s.strip() for s in corrected_batch]


# ---------- Correction Function ----------

In [11]:
# def correct_vietnamese_spelling(text, row_id, col_name, debug_print=False):
#     original_text = "" if pd.isna(text) else str(text).strip()
#     if not original_text:
#         return text

#     cache_key = (col_name, original_text)
#     if cache_key in correction_cache:
#         return correction_cache[cache_key]

#     prompt = make_correction_prompt(original_text)
#     corrected_text = generate_from_model(prompt)

#     # --- GHI CHÚ ---
#     # Bây giờ bạn đã thêm prefix vào đầu vào,
#     # rất có thể mô hình sẽ không tạo ra prefix này ở đầu ra nữa.
#     # Bạn hãy theo dõi: nếu prefix này không còn xuất hiện ở đầu ra
#     # (trong phần DEBUG), bạn có thể xóa bỏ 3 dòng code dưới đây.
#     prefix_to_remove = "Sửa lỗi chính tả và ngữ pháp:"
#     if corrected_text.startswith(prefix_to_remove):
#         corrected_text = corrected_text[len(prefix_to_remove) :].strip()

#     accepted = False
#     reason = "not_changed"
#     final_text = original_text

#     if corrected_text and corrected_text.lower() != original_text.lower():
#         is_acceptable = False

#         # --- Lớp 1: Kiểm tra tương đồng và độ dài ---
#         orig_base = remove_diacritics_and_punct(original_text)
#         corr_base = remove_diacritics_and_punct(corrected_text)
#         base_sim = string_similarity(orig_base, corr_base)

#         if col_name == "context":
#             if orig_base != corr_base:
#                 reason = "base_text_altered_in_context"
#             elif (
#                 abs(len(corrected_text) - len(original_text))
#                 / max(1, len(original_text))
#                 > LENGTH_CHANGE_ALLOWED_RATIO
#             ):
#                 reason = f"length_changed_too_much"
#             else:
#                 is_acceptable = True
#         else:
#             if base_sim < STRICT_BASE_SIMILARITY_THRESHOLD:
#                 reason = f"base_similarity_too_low_{base_sim:.2f}"
#             elif (
#                 string_similarity(original_text, corrected_text)
#                 < ACCEPT_SIMILARITY_THRESHOLD
#             ):
#                 reason = f"low_similarity_{string_similarity(original_text, corrected_text):.2f}"
#             elif (
#                 abs(len(corrected_text) - len(original_text))
#                 / max(1, len(original_text))
#                 > LENGTH_CHANGE_ALLOWED_RATIO
#             ):
#                 reason = f"length_changed_too_much"
#             else:
#                 is_acceptable = True

#         # --- Lớp 2: Kiểm tra sự thật (Fact-Checking) ---
#         if is_acceptable:
#             original_numbers = extract_numbers(original_text)
#             corrected_numbers = extract_numbers(corrected_text)

#             original_nouns = extract_proper_nouns(original_text)
#             corrected_nouns = extract_proper_nouns(corrected_text)

#             if not corrected_numbers.issubset(original_numbers):
#                 reason = (
#                     f"new_number_introduced: {corrected_numbers - original_numbers}"
#                 )
#                 is_acceptable = False
#             elif not corrected_nouns.issubset(original_nouns):
#                 reason = (
#                     f"new_proper_noun_introduced: {corrected_nouns - original_nouns}"
#                 )
#                 is_acceptable = False

#         if is_acceptable:
#             accepted = True
#             final_text = corrected_text
#             reason = "accepted_change"

#     if debug_print:
#         print("\n--- DEBUG SAMPLE ---")
#         print(f"ID: {row_id} | COLUMN: {col_name}")
#         print(f"ORIGINAL: \n{original_text}")
#         print(f"CORRECTED: \n{corrected_text}")

#         if corrected_text and corrected_text.lower() != original_text.lower():
#             print("--- METRICS & THRESHOLDS ---")
#             direct_sim = string_similarity(original_text, corrected_text)
#             base_sim = string_similarity(
#                 remove_diacritics_and_punct(original_text),
#                 remove_diacritics_and_punct(corrected_text),
#             )
#             length_change = abs(len(corrected_text) - len(original_text)) / max(
#                 1, len(original_text)
#             )

#             if col_name == "context":
#                 print(f"Rule for '{col_name}': Base text must be identical.")
#                 print(
#                     f"  - Base Text Match: {remove_diacritics_and_punct(original_text) == remove_diacritics_and_punct(corrected_text)}"
#                 )
#             else:
#                 print(f"Rules for '{col_name}':")
#                 print(
#                     f"  - Base Similarity: {base_sim:.4f} (Threshold >= {STRICT_BASE_SIMILARITY_THRESHOLD})"
#                 )
#                 print(
#                     f"  - Direct Similarity: {direct_sim:.4f} (Threshold >= {ACCEPT_SIMILARITY_THRESHOLD})"
#                 )

#             print(
#                 f"  - Length Change Ratio: {length_change:.4f} (Threshold <= {LENGTH_CHANGE_ALLOWED_RATIO})"
#             )

#             print("--- FACT CHECKING ---")
#             print(f"  - Original Numbers: {extract_numbers(original_text)}")
#             print(f"  - Corrected Numbers: {extract_numbers(corrected_text)}")
#             print(f"  - Original Nouns: {extract_proper_nouns(original_text)}")
#             print(f"  - Corrected Nouns: {extract_proper_nouns(corrected_text)}")

#         else:
#             print("--- METRICS & THRESHOLDS ---")
#             print("No change proposed by model.")

#         print("--- DECISION ---")
#         print(f"RESULT: {'ACCEPTED' if accepted else 'REJECTED'}")
#         if not accepted:
#             print(f"REASON: {reason}")
#         print(f"FINAL TEXT: \n{final_text}")
#         print("-" * 20)

#     correction_cache[cache_key] = final_text
#     return final_text


# ===== THAY ĐỔI: Hàm này giờ chỉ làm "VALIDATE" (Kiểm tra) => sử dụng theo batch =====

In [12]:
# ===== THAY ĐỔI: Hàm này giờ chỉ làm "VALIDATE" (Kiểm tra) => sử dụng theo batch =====
# Nó không còn gọi model nữa, mà chỉ so sánh 2 văn bản
def validate_correction(
    original_text, corrected_text_from_model, row_id, col_name, debug_print=False
):
    original_text = "" if pd.isna(original_text) else str(original_text).strip()
    corrected_text = corrected_text_from_model  # Lấy từ kết quả batch

    if not original_text:
        return original_text, False  # Trả về text gốc, và False (không chấp nhận)

    # --- Xử lý prefix (Vô hiệu hóa vì model BART không dùng) -> ViT5 vẫn dùng ---
    prefix_to_remove = "Sửa lỗi chính tả và ngữ pháp:"
    if corrected_text.startswith(prefix_to_remove):
        corrected_text = corrected_text[len(prefix_to_remove) :].strip()

    accepted = False
    reason = "not_changed"
    final_text = original_text

    if corrected_text and corrected_text.lower() != original_text.lower():
        is_acceptable = False
        # --- Lớp 1: Kiểm tra tương đồng và độ dài ---
        orig_base = remove_diacritics_and_punct(original_text)
        corr_base = remove_diacritics_and_punct(corrected_text)
        base_sim = string_similarity(orig_base, corr_base)

        if col_name == "context":
            if orig_base != corr_base:
                reason = "base_text_altered_in_context"
            elif (
                abs(len(corrected_text) - len(original_text))
                / max(1, len(original_text))
                > LENGTH_CHANGE_ALLOWED_RATIO
            ):
                reason = f"length_changed_too_much"
            else:
                is_acceptable = True
        else:
            if base_sim < STRICT_BASE_SIMILARITY_THRESHOLD:
                reason = f"base_similarity_too_low_{base_sim:.2f}"
            elif (
                string_similarity(original_text, corrected_text)
                < ACCEPT_SIMILARITY_THRESHOLD
            ):
                reason = f"low_similarity_{string_similarity(original_text, corrected_text):.2f}"
            elif (
                abs(len(corrected_text) - len(original_text))
                / max(1, len(original_text))
                > LENGTH_CHANGE_ALLOWED_RATIO
            ):
                reason = f"length_changed_too_much"
            else:
                is_acceptable = True

        # --- Lớp 2: Kiểm tra sự thật (Fact-Checking) ---
        if is_acceptable:
            original_numbers = extract_numbers(original_text)
            corrected_numbers = extract_numbers(corrected_text)

            original_nouns = extract_proper_nouns(original_text)
            corrected_nouns = extract_proper_nouns(corrected_text)

            if not corrected_numbers.issubset(original_numbers):
                reason = (
                    f"new_number_introduced: {corrected_numbers - original_numbers}"
                )
                is_acceptable = False

            # Kiểm tra xem có danh từ GỐC nào bị xóa mất không
            elif not original_nouns.issubset(corrected_nouns):
                # Chỉ áp dụng nếu original_nouns không rỗng
                if original_nouns:
                    reason = f"proper_noun_deleted: {original_nouns - corrected_nouns}"
                    is_acceptable = False

            elif not corrected_nouns.issubset(original_nouns):
                reason = (
                    f"new_proper_noun_introduced: {corrected_nouns - original_nouns}"
                )
                is_acceptable = False

        if is_acceptable:
            accepted = True
            final_text = corrected_text
            reason = "accepted_change"

    if debug_print:
        print("\n--- DEBUG SAMPLE ---")
        print(f"ID: {row_id} | COLUMN: {col_name}")
        print(f"ORIGINAL: \n{original_text}")
        print(f"CORRECTED (from model): \n{corrected_text}")  # Đã đổi tên biến

        if corrected_text and corrected_text.lower() != original_text.lower():
            print("--- METRICS & THRESHOLDS ---")
            # ... (toàn bộ logic debug của bạn giữ nguyên) ...
            direct_sim = string_similarity(original_text, corrected_text)
            base_sim = string_similarity(
                remove_diacritics_and_punct(original_text),
                remove_diacritics_and_punct(corrected_text),
            )
            length_change = abs(len(corrected_text) - len(original_text)) / max(
                1, len(original_text)
            )
            if col_name == "context":
                print(f"Rule for '{col_name}': Base text must be identical.")
                print(
                    f"  - Base Text Match: {remove_diacritics_and_punct(original_text) == remove_diacritics_and_punct(corrected_text)}"
                )
            else:
                print(f"Rules for '{col_name}':")
                print(
                    f"  - Base Similarity: {base_sim:.4f} (Threshold >= {STRICT_BASE_SIMILARITY_THRESHOLD})"
                )
                print(
                    f"  - Direct Similarity: {direct_sim:.4f} (Threshold >= {ACCEPT_SIMILARITY_THRESHOLD})"
                )
            print(
                f"  - Length Change Ratio: {length_change:.4f} (Threshold <= {LENGTH_CHANGE_ALLOWED_RATIO})"
            )
            print("--- FACT CHECKING ---")
            print(f"  - Original Numbers: {extract_numbers(original_text)}")
            print(f"  - Corrected Numbers: {extract_numbers(corrected_text)}")
            print(f"  - Original Nouns: {extract_proper_nouns(original_text)}")
            print(f"  - Corrected Nouns: {extract_proper_nouns(corrected_text)}")
        else:
            print("--- METRICS & THRESHOLDS ---")
            print("No change proposed by model.")

        print("--- DECISION ---")
        print(f"RESULT: {'ACCEPTED' if accepted else 'REJECTED'}")
        if not accepted:
            print(f"REASON: {reason}")
        print(f"FINAL TEXT: \n{final_text}")
        print("-" * 20)

    # Trả về text cuối cùng VÀ cờ (flag)
    return final_text, accepted



# ---------------- Main loop ----------------

In [13]:
if not os.path.exists(INPUT_CSV):
    raise FileNotFoundError(f"Không tìm thấy file '{INPUT_CSV}'.")


In [14]:
df = pd.read_csv(INPUT_CSV)
df_global = df
corrected_ids = set()
df.head(10)


,id,context,prompt,response,predict_label
0,b709059b-b3b6-4ac2-bb88-2c794e2cc219,"Putin ngày 14 tháng 10 năm 2009, đưa ra đề ngh...",Ý nhĩa cũa viẹc tổ chưc cuôc thi Intervision l...,"Cuộc thi Intervision, dự kiến lấy cảm hứng từ ...",NaN
1,7dc35ef5-c4b7-4538-ab90-627b9cbd896e,"Thông qua những quyết nghị này, Viện nguyên lã...",Quyền lực của Viện nguyên lão đã giảm mạnh khi...,"Sai. Thực tế, Viện nguyên lão vẫn duy trì quyề...",NaN
2,cfdfa010-f61c-4845-91c9-23f79be2b88b,Nơi cư ngụ truyền thống Mông Cổ được gọi là mộ...,Theo quan điểm của ai thì kiến trúc truyền thố...,"Ngoài N. Chultem, một số học giả như nhà sử họ...",NaN
3,31b33c97-2f59-4e72-8707-f47de204d7f9,Là thành phố thủ đô và có vị trí ở khu vực tru...,Hà Nội hiện nay có tuyến đường sắt quốc tế kết...,Hà Nội hiện nay có tuyến đường sắt quốc tế trự...,NaN
4,a2c83a00-e8b7-4236-86ce-5e0104df074a,Từ 1974 đến nay tốc độ phát triển tuy chậm lại...,Câu hỏi gài bẫy: Tiền đề để trở thành một quốc...,Tiền đề để Nhật Bản trở thành quốc gia cho vay...,NaN
5,c5972f72-948c-4907-a0d9-5a5077d263eb,"Vào năm 1580, các tu sĩ dòng Tên thiết lập Mai...",Các nhà thần học và khoa học muốn tìm đến S.J ...,Các nhà thần học và khoa học muốn tìm đến S.J ...,NaN
6,c9cb5df4-6c6a-44db-977f-b5a9d59926f0,"Hiện nay, do đời sống được nâng cao hơn, cơ cấ...","Các gia đình, nhà hàng hiện nay trình bày chén...",Các gia đình và nhà hàng hiện nay thường san r...,NaN
7,87ae8fca-aa1d-4ba4-bebe-fed2c811dd29,Lãnh thổ Thành Vatican là bộ phận của Mons Vat...,"Trước năm 1929, khu vực lãnh thổ Thành Vatican...","Trước năm 1929, khu vực lãnh thổ Thành Vatican...",NaN
8,faa318fc-0679-486c-98fd-54823a2374f9,"Vì là vị hoàng đế đầu tiên, Augustus đã đảm nh...","Mặc dù Augustus đã tự gọi mình là ""Đệ nhất côn...",Augustus đã ngay lập tức xóa bỏ các thể chế cộ...,NaN
9,1d0a69e2-146d-492a-8c4a-3f77b7395c6a,"Ngày 7/9/1945, tờ Báo Tranh đấu, cơ quan ngôn ...",Ý nghĩa của thông cáo do Trần Văn Giàu đưa ra ...,Thông cáo của Trần Văn Giàu không chỉ kêu gọi ...,NaN


In [15]:
# # ===== THAY ĐỔI: Xử lý theo cột, for infernce =====
# for index, row in tqdm(df.iterrows(), total=df.shape[0], desc="Processing"):
#     current_id = row["id"]
#     for col in COLUMNS_TO_FIX:
#         original_text = row[col]
#         debug_flag = index < SAMPLE_DEBUG_N
#         corrected_text = correct_vietnamese_spelling(
#             original_text, current_id, col, debug_print=debug_flag
#         )
#         if str(original_text) != str(corrected_text):
#             corrected_ids.add(current_id)
#             df.at[index, col] = corrected_text

# print(f"\nĐã hoàn thành. Có {len(corrected_ids)} dòng được thay đổi.")


In [16]:
# ===== THAY ĐỔI: Xử lý theo cột, với batch inference =====
# Chỉ xử lý các cột được chỉ định
for col_to_fix in COLUMNS_TO_FIX:
    print(f"\n--- Processing column: {col_to_fix} ---")

    # Lấy toàn bộ cột ra thành 1 list
    original_texts = df[col_to_fix].fillna("").astype(str).tolist()

    # 1. Tạo prompts cho tất cả
    # (Hàm này chạy nhanh, không cần batch)
    prompts_to_model = [make_correction_prompt(t) for t in original_texts]

    # 2. Bước INFERENCE (Xử lý hàng loạt bằng model)
    model_outputs = []
    print(
        f"Running inference on {len(prompts_to_model)} items (Batch size: {BATCH_SIZE})..."
    )

    # Tqdm cho vòng lặp batch
    for i in tqdm(
        range(0, len(prompts_to_model), BATCH_SIZE), desc=f"Inference {col_to_fix}"
    ):
        # Cắt ra 1 batch
        batch_prompts = prompts_to_model[i : i + BATCH_SIZE]

        # Gọi hàm xử lý batch mới
        corrected_batch = generate_batch_from_model(batch_prompts)

        # Thêm kết quả vào list tổng
        model_outputs.extend(corrected_batch)

    # 3. Bước VALIDATION (Kiểm tra và cập nhật)
    # Lúc này, `original_texts` và `model_outputs` là 2 list
    # có cùng độ dài và thứ tự 100% khớp nhau.
    print("Validating results and updating dataframe...")
    final_texts_for_column = []

    # Tqdm cho vòng lặp validation
    iterator = zip(df.iterrows(), original_texts, model_outputs)
    for (index, row), original, corrected in tqdm(
        iterator, total=len(df), desc=f"Validate {col_to_fix}"
    ):

        current_id = row["id"]
        debug_flag = index < SAMPLE_DEBUG_N

        # Gọi hàm validate mới
        final_text, accepted = validate_correction(
            original, corrected, current_id, col_to_fix, debug_print=debug_flag
        )

        if accepted:
            corrected_ids.add(current_id)

        # Cập nhật trực tiếp vào dataframe
        # (Cách này an toàn cho Ctrl+C handler)
        df.at[index, col_to_fix] = final_text



--- Processing column: prompt ---
Running inference on 1000 items (Batch size: 32)...


Inference prompt: 100%|██████████| 32/32 [01:16<00:00,  2.38s/it]


Validating results and updating dataframe...


Validate prompt: 100%|██████████| 1000/1000 [00:00<00:00, 13155.92it/s]


--- DEBUG SAMPLE ---
ID: b709059b-b3b6-4ac2-bb88-2c794e2cc219 | COLUMN: prompt
ORIGINAL: 
Ý nhĩa cũa viẹc tổ chưc cuôc thi Intervision là gì z?
CORRECTED (from model): 
Ý nghĩa của việc tổ chức cuộc thi Intervision là gì?
--- METRICS & THRESHOLDS ---
Rules for 'prompt':
  - Base Similarity: 0.9709 (Threshold >= 0.92)
  - Direct Similarity: 0.8952 (Threshold >= 0.88)
  - Length Change Ratio: 0.0189 (Threshold <= 0.2)
--- FACT CHECKING ---
  - Original Numbers: set()
  - Corrected Numbers: set()
  - Original Nouns: {'intervision'}
  - Corrected Nouns: {'intervision'}
--- DECISION ---
RESULT: ACCEPTED
FINAL TEXT: 
Ý nghĩa của việc tổ chức cuộc thi Intervision là gì?
--------------------

--- DEBUG SAMPLE ---
ID: 7dc35ef5-c4b7-4538-ab90-627b9cbd896e | COLUMN: prompt
ORIGINAL: 
Quyền lực của Viện nguyên lão đã giảm mạnh khi lãnh thổ La Mã cổ đại được mở rộng, vì các chánh quan tòa hoàn toàn tự chủ trong việc quản lý các tỉnh mà không cần sự giám sát từ Viện nguyên lão, đúng hay sai?
CORREC

In [17]:
if os.path.exists(OUTPUT_CSV):
    print(f"\n[WARNING] File '{OUTPUT_CSV}' đã tồn tại và sẽ bị ghi đè.")

# Ghi file (dù có tồn tại hay không)
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"\n✅ Output đã được lưu vào {OUTPUT_CSV}")



[WARNING] File '/home/guest/Projects/CS221/data/preprocessed/vihallu-test-ViT5_Vietnamese_Correction-prompt-preprocessed.csv' đã tồn tại và sẽ bị ghi đè.

✅ Output đã được lưu vào /home/guest/Projects/CS221/data/preprocessed/vihallu-test-ViT5_Vietnamese_Correction-prompt-preprocessed.csv
